In [ ]:
import pickle
import numpy as np
import tensorflow as tf



# Path to your data file
data_path = '/content/drive/MyDrive/project/RML2016.10a_dict.pkl'

# Load data
with open(data_path, 'rb') as file:
    Xd = pickle.load(file, encoding='iso-8859-1')

# Extract mods and snrs
mods, snrs1 = [sorted(list(set([k[j] for k in Xd.keys()]))) for j in [0,1]]

X = []
lbl = []
train_idx = []
val_idx = []
np.random.seed(2016)
a = 0 #a between 0and 220
#
snrs1=[-20,-18,-16,-14,-12,-10,-8,-6,-4,-2,0,2,4,6,8,10,12,14,16,18]
# Populate X and lbl
for mod in mods:
    for snr in snrs1:
        X.append(Xd[(mod, snr)])
        for i in range(Xd[(mod, snr)].shape[0]):#Xd[(mod, snr)].shape[0]=1000
            lbl.append((mod, snr))
        train_idx += list(np.random.choice(range(a*1000, (a+1)*1000), size=600, replace=False))
        val_idx += list(np.random.choice(list(set(range(a*1000, (a+1)*1000)) - set(train_idx)), size=200, replace=False))
        a += 1

# Stack X
X = np.vstack(X)

# Define test_idx
n_examples = X.shape[0]
test_idx = list(set(range(0, n_examples)) - set(train_idx) - set(val_idx))
np.random.shuffle(train_idx)
np.random.shuffle(val_idx)
np.random.shuffle(test_idx)

# Split into train, val, and test sets
X_train = X[train_idx]
X_val = X[val_idx]
X_test = X[test_idx]

# Define to_onehot function
def to_onehot(yy):
    yy1 = np.zeros([len(yy), len(mods)])
    yy1[np.arange(len(yy)), yy] = 1
    return yy1

# Generate Y_train, Y_val, and Y_test
Y_train = to_onehot(list(map(lambda x: mods.index(lbl[x][0]), train_idx)))
Y_val = to_onehot(list(map(lambda x: mods.index(lbl[x][0]), val_idx)))
Y_test = to_onehot(list(map(lambda x: mods.index(lbl[x][0]), test_idx)))

X_train = X_train.swapaxes(2, 1)
X_val = X_val.swapaxes(2, 1)
X_test = X_test.swapaxes(2, 1)


# Print shapes to verify

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_val.shape)
print(Y_test.shape)


In [ ]:
from tensorflow.keras.layers import Input,GRU, Conv2D, Dense,Add,Subtract, Dropout,Multiply, BatchNormalization, Activation,Lambda, Flatten, Reshape, MaxPooling2D, concatenate
from keras.callbacks import EarlyStopping
X1_train=X_train[:,:,0]
X1_test=X_test[:,:,0]
X1_val=X_val[:,:,0]
X2_train=X_train[:,:,1]
X2_test=X_test[:,:,1]
X2_val=X_val[:,:,1]
X_train=np.expand_dims(X_train,axis=3)
X_test=np.expand_dims(X_test,axis=3)
X_val=np.expand_dims(X_val,axis=3)
print(X1_train.shape)

print(X_test.shape)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
X_train2 = tf.reshape(X_train, (132000, 256))
X_val2 = tf.reshape(X_val, (44000, 256),)
X_test2 = tf.reshape(X_test, (44000, 256),)
print(X_train2.shape)

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense,Dropout
import tensorflow as tf
def create_fcn(input_dim):
    inputs = Input(shape=(input_dim,))
    hidden1 = Dense(500, activation='relu')(inputs)
    dropout1 = Dropout(0.2)(hidden1)
    hidden2 = Dense(600, activation='relu')(dropout1)
    dropout2 = Dropout(0.2)(hidden2)
    hidden3= Dense(300, activation='relu')(dropout2)
    dropout3 = Dropout(0.2)(hidden3)
    outputs = Dense(1, activation='linear')(dropout3)
    model = Model(inputs=inputs, outputs=outputs)
    return model

In [ ]:
model=create_fcn(256)

In [ ]:
model.load_weights('/content/drive/MyDrive/Colab Notebooks/weight80.h5')


In [ ]:
combined_array= np.concatenate((X_train2, X_val2, X_test2), axis=0)

In [ ]:
y=model.predict(combined_array)


In [ ]:
print(max(y))


In [ ]:
from math import pi  # Import pi from the math library

sequence = np.arange(1, 129)

# Use broadcasting to multiply each value in y with the sequence
result_array = (y* sequence*pi)/(8*100000)

print(result_array.shape)

In [ ]:
cos=np.cos(result_array)


In [ ]:
print(cos.shape)

In [ ]:
sin=np.sin(result_array)

In [ ]:
combined1 = np.vstack((X1_train, X1_val,X1_test))

print(combined1.shape)  # This should give (220000, 256)

In [ ]:
combined2 = np.vstack((X2_train, X2_val,X2_test))

print(combined2.shape)  #

In [ ]:
from tensorflow.keras.layers import Input,Activation, Conv1D,concatenate, MaxPooling1D, Dropout, GRU, Flatten, Dense, Concatenate, Lambda,Multiply, Add, Subtract, Reshape
from tensorflow.keras.models import Model
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, Dense, Flatten, Reshape, Concatenate
from tensorflow.keras.models import Model

def CNN(input_shape=(128, 2), input_shape2=(128, 1), classes=11):

    # Define input layers
    dr = 0.5 # dropout rate (%)
    input = Input(shape=input_shape + (1,), name='input1')  # Shape will be (None, 2, 128, 1)
    input1 = Input(input_shape2, name='input2')
    input2 = Input(input_shape2, name='input3')



    # spatial feature
    x3 = Conv2D(75, (8,2), padding='valid', activation="relu", name="conv1_1", kernel_initializer='glorot_uniform')(
        input)
    x3 = Conv2D(25, (5,1), padding='valid', activation="relu", name="conv1_2", kernel_initializer='glorot_uniform')(
        x3)
    # temporal feature
    x4 = Reshape(target_shape=((117,25)), name='reshape4')(x3)
    x4= GRU(units=128)(x4)

    x = Dense(classes, activation='softmax', name='softmax')(x4)


    model = Model(inputs = [input,input1,input2], outputs=x)



    return model




In [ ]:
from keras.optimizers import Adam
model=CNN()
from keras.optimizers import AdamW
from keras.optimizers.schedules import ExponentialDecay



model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
array1 = combined1 *cos
array2 = combined2*sin
array3 = combined2*cos
array4 = combined1*sin
I=array1+array2
Q=array3-array4

In [ ]:
print(X2_train.shape)

In [ ]:
train_new1 = I[:132000]    # First 132000 rows
val_new1 = I[132000:176000]  # Next 44000 rows
test_new1 = I[176000:]    # Remaining rows

print(train_new1.shape)  # Should print (132000, 128)
print(val_new1.shape)    # Should print (44000, 128)
print(test_new1.shape)   # Should print (44000, 128)

In [ ]:
train_new2 = Q[:132000]    # First 132000 rows
val_new2 = Q[132000:176000]  # Next 44000 rows
test_new2 = Q[176000:]    # Remaining rows

print(train_new2.shape)  # Should print (132000, 128)
print(val_new2.shape)    # Should print (44000, 128)
print(test_new2.shape)   # Should print (44000, 128)

In [ ]:
import numpy as np

X_train_new = np.stack((train_new1, train_new2), axis=-1)
X_val_new = np.stack((val_new1, val_new2), axis=-1)
X_test_new = np.stack((test_new1, test_new2), axis=-1)




In [ ]:
X_train_new.shape

In [ ]:
model1=CNN()

In [ ]:
model1.load_weights('/content/drive/MyDrive/Colab Notebooks/mmmmm1.keras')

In [ ]:
from keras import callbacks
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Reshape,GRU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import callbacks
from tensorflow.keras.callbacks import ModelCheckpoint,ReduceLROnPlateau,EarlyStopping

modtype=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM']
filepath =('/content/drive/MyDrive/Colab Notebooks/mmmmm2.keras')
history=model.fit([X_train_new, train_new1, train_new2], Y_train, epochs=100, batch_size=64,
         validation_data=([X_val_new, val_new1, val_new2], Y_val),callbacks = [
                ModelCheckpoint(filepath, monitor='val_loss', verbose=1, save_best_only=True, mode='auto'),
                ReduceLROnPlateau(monitor='val_loss',factor=0.5,verbose=1,patince=5,min_lr=0.0000001),
                EarlyStopping(monitor='val_loss', patience=5, verbose=1, mode='auto'),
                ])


In [ ]:
train_loss = history.history['loss']
val_loss = history.history['val_loss']

In [ ]:
train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(train_loss) + 1)

# Plotting loss
plt.figure(figsize=(6, 3))
plt.plot(epochs, train_loss, 'r', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Plotting MAE if available
if 'accuracy' in history.history:
    plt.figure(figsize=(6, 3))
    plt.plot(epochs, train_accuracy, 'r', label='Training accuracy')
    plt.plot(epochs, val_accuracy, 'b', label='Validation accuracy')
    plt.title('Training and validation accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('accuracy')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import csv

def plot_confusion_matrix(cm, title='Confusion matrix', cmap=plt.get_cmap("PuRd"), labels=[], save_filename=None):
    plt.figure(figsize=(6, 6))
    plt.imshow(cm * 100, interpolation='nearest', cmap=cmap)
    plt.colorbar()
    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=90, size=12)
    plt.yticks(tick_marks, labels, size=12)

    for i in range(len(tick_marks)):
        for j in range(len(tick_marks)):
            value = int(np.around(cm[i, j] * 100))
            if i != j:
                text = plt.text(j, i, f"{value}%", ha="center", va="center", fontsize=10)
            elif i == j:
                if value == 100:
                    text = plt.text(j, i, f"{value}%", ha="center", va="center", fontsize=7, color="white")
                else:
                    text = plt.text(j, i, f"{value}%", ha="center", va="center", fontsize=10, color="white")

    plt.tight_layout()
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')



In [ ]:
def calculate_confusion_matrix(Y,Y_hat,classes):
    n_classes = len(classes)
    conf = np.zeros([n_classes,n_classes])
    confnorm = np.zeros([n_classes,n_classes])

    for k in range(0,Y.shape[0]):
        i = list(Y[k,:]).index(1)
        j = int(np.argmax(Y_hat[k,:]))
        conf[i,j] = conf[i,j] + 1

    for i in range(0,n_classes):
        confnorm[i,:] = conf[i,:] / np.sum(conf[i,:])
    # print(confnorm)

    right = np.sum(np.diag(conf))
    wrong = np.sum(conf) - right
    return confnorm,right,wrong

In [ ]:
classes=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM']
plt.figure(figsize=(12, 10))
test_Y_hat = model1.predict([X_test_new, test_new1, test_new2], batch_size=32)
confnorm, _, _ = calculate_confusion_matrix(Y_test, test_Y_hat, classes)
plot_confusion_matrix(confnorm, labels=classes)
acc = {}
acc_mod_snr = np.zeros( (len(classes),len(snrs1)) )
i = 0
for snr in snrs1:
        test_SNRs = [lbl[x][1] for x in test_idx]
        test_X_i=X_test_new[np.where(np.array(test_SNRs) == snr)]
        test_X1_i = test_new1[np.where(np.array(test_SNRs) == snr)]
        test_X2_i = test_new2[np.where(np.array(test_SNRs) == snr)]
        test_Y_i = Y_test[np.where(np.array(test_SNRs) == snr)]


        test_Y_i_hat = model1.predict([test_X_i,test_X1_i,test_X2_i])

        confnorm_i,cor,ncor = calculate_confusion_matrix(test_Y_i,test_Y_i_hat,classes)
        acc[snr] = 1.0 * cor / (cor + ncor)
        result = cor / (cor + ncor)

        plot_confusion_matrix(confnorm_i, labels=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM'])
        plt.show()
        acc_mod_snr[:,i] = np.round(np.diag(confnorm_i)/np.sum(confnorm_i,axis=1),3)
        i = i +1






In [ ]:
import numpy as np
import matplotlib.pyplot as plt

num_mod = 11

plt.figure(figsize=(8, 6))
plt.xlabel("Signal to Noise Ratio")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy for Each Mod")

for i in range(0, num_mod):
    plt.plot(snrs1, acc_mod_snr[i], label=classes[i])

# Add legend, grid, save, and show plot
plt.legend()
plt.grid()
plt.savefig('marah.png')
plt.show()
plt.close()


In [ ]:
# Plot Accuracy vs. SNR
plt.figure(figsize=(10, 6))
plt.plot(snrs1, list(map(lambda x: acc[x], snrs1)), marker='o', linestyle='-', color='b', label='Accuracy')

for snr, accuracy in zip(snrs1, list(map(lambda x: acc[x], snrs1))):
    plt.annotate(f'{accuracy:.2f}',
                 (snr, accuracy),
                 textcoords="offset points",
                 xytext=(0,10),
                 ha='center')

plt.xlabel("Signal to Noise Ratio (SNR)")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy vs. SNR")
plt.grid(True)
plt.savefig('figure1.png')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import csv
#sspecific snr

classes = mods
modtype=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM']
test_Y_hat = model1.predict([X_test_new, test_new1, test_new2])
confnorm, _, _ = calculate_confusion_matrix(Y_test, test_Y_hat, classes)
acc2 = {}
acc_mod_snr = np.zeros( (len(classes),len(snrs1)) )
i = 0
snr =0
test_SNRs = [lbl[x][1] for x in test_idx]
test_X_i=X_test_new[np.where(np.array(test_SNRs) == snr)]
test_X1_i = test_new1[np.where(np.array(test_SNRs) == snr)]
test_X2_i = test_new2[np.where(np.array(test_SNRs) == snr)]
test_Y_i = Y_test[np.where(np.array(test_SNRs) == snr)]




test_Y_i_hat = model1.predict([test_X_i,test_X1_i,test_X2_i])


confnorm_i,cor,ncor = calculate_confusion_matrix(test_Y_i,test_Y_i_hat,classes)

plt.figure(figsize=(4, 4))
plot_confusion_matrix(confnorm_i, labels=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM'])

plt.title(f'Confusion Matrix at SNR={snr} dB')
plt.savefig('mm1')







In [ ]:
import numpy as np
import matplotlib.pyplot as plt



plt.figure(figsize=(10, 6))
count, bins, _ = plt.hist(y, bins=30, density=True, color='blue', edgecolor='black')

plt.xlabel('Frequency Offset(Hz)')
plt.ylabel('Signal Count (Normalized)')
plt.title('Histogram of Frequency Offset')

plt.show()


In [ ]:
std_deviation_value = np.std(y*8/100000)


In [ ]:
std_deviation_value

In [ ]:
model2=CNN()
model2.load_weights('/content/drive/MyDrive/CNN_2D_weights(1).h5')

In [ ]:
classes=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM']
plt.figure(figsize=(12, 10))
confnorm, _, _ = calculate_confusion_matrix(Y_test, test_Y_hat, classes)
#snrs1=snrs
plot_confusion_matrix(confnorm, labels=classes)
acc1 = {}
acc_mod_snr = np.zeros( (len(classes),len(snrs1)) )
i = 0
for snr in snrs1:
        test_SNRs1 = [lbl[x][1] for x in test_idx]
        test_X_i=X_test[np.where(np.array(test_SNRs1) == snr)]
        test_X1_i = X1_test[np.where(np.array(test_SNRs1) == snr)]
        test_X2_i = X2_test[np.where(np.array(test_SNRs1) == snr)]
        test_Y_i = Y_test[np.where(np.array(test_SNRs1) == snr)]


        test_Y_i_hat = model2.predict([test_X_i,test_X1_i,test_X2_i])

        confnorm_i,cor,ncor = calculate_confusion_matrix(test_Y_i,test_Y_i_hat,classes)
        acc1[snr] = 1.0 * cor / (cor + ncor)
        result = cor / (cor + ncor)
        plot_confusion_matrix(confnorm_i, labels=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM'])
        plt.show()

        acc_mod_snr[:,i] = np.round(np.diag(confnorm_i)/np.sum(confnorm_i,axis=1),3)
        i = i +1


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

num_mod = 11

plt.figure(figsize=(8, 6))
plt.xlabel("Signal to Noise Ratio")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy for Each Mod")

for i in range(0, num_mod):
    plt.plot(snrs1, acc_mod_snr[i], label=classes[i])

# Add legend, grid, save, and show plot
plt.legend()
plt.grid()
plt.savefig('marah1.png')
plt.show()
plt.close()


In [ ]:
# Plot Accuracy vs. SNR
plt.figure(figsize=(10, 6))
plt.plot(snrs1, list(map(lambda x: acc1[x], snrs1)), marker='o', linestyle='-', color='b', label='Accuracy')

for snr, accuracy in zip(snrs1, list(map(lambda x: acc1[x], snrs1))):
    plt.annotate(f'{accuracy:.2f}',
                 (snr, accuracy),
                 textcoords="offset points",
                 xytext=(0,10),
                 ha='center')

plt.xlabel("Signal to Noise Ratio (SNR)")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy vs. SNR")
plt.grid(True)
plt.savefig('figure2.png')
plt.show()

In [ ]:
# Plot Accuracy vs. SNR
plt.figure(figsize=(10, 6))
plt.plot(snrs1, list(map(lambda x: acc[x], snrs1)), marker='o', linestyle='-', color='orange', label='after')

# Annotate each point with its value
for snr, accuracy in zip(snrs1, list(map(lambda x: acc[x], snrs1))):
    plt.annotate(f'{accuracy:.2f}',
                 (snr, accuracy),
                 textcoords="offset points",
                 xytext=(0,10),
                 ha='center')
plt.plot(snrs1, list(map(lambda x: acc1[x], snrs1)), marker='o', linestyle='-', color='b', label='before')

# Annotate each point with its value
for snr, accuracy in zip(snrs1, list(map(lambda x: acc1[x], snrs1))):
    plt.annotate(f'{accuracy:.2f}',
                 (snr, accuracy),
                 textcoords="offset points",
                 xytext=(0,10),
                 ha='center')
# Adding labels and title
plt.xlabel("Signal to Noise Ratio (SNR)")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy vs. SNR")
plt.grid(True)
plt.legend()
# Save the figure to a file and show it
plt.savefig('figure1.png')  # Save the figure to a file
plt.show()